# OpenPlaque — LAD Distal Endpoint Continuation v1
Independent source-CCTA test of whether the blind backbone finding is actually a distal continuation of the frozen LAD. Run with **Runtime → Run all**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, sys
os.chdir('/content')
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT = DRIVE_ROOT / 'LAD_Distal_Endpoint_Continuation_v1'
REUSE_EXISTING_OUTPUT = False
if OUTPUT.exists() and not REUSE_EXISTING_OUTPUT:
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'notebook_started.json').write_text(json.dumps({'status':'STARTED','notebook':'LAD distal endpoint continuation v1'}, indent=2))
print('Output:', OUTPUT)

In [ ]:
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
BRANCH = 'lad-distal-endpoint-continuation-from-main'
PINNED_SCIENCE_COMMIT = '3bc1fb900a3251bdd76ebf7fc57f700d65856c29'
repo = '/content/OpenPlaque_lad_distal_v1'
os.chdir('/content')
if os.path.exists(repo):
    shutil.rmtree(repo)
!git clone --depth 20 --branch "$BRANCH" https://github.com/pazzani/OpenPlaque.git "$repo"
!git -C "$repo" checkout --detach "$PINNED_SCIENCE_COMMIT"
HEAD = get_ipython().getoutput(f'git -C {repo} rev-parse HEAD')[-1].strip()
MB = get_ipython().getoutput(f'git -C {repo} merge-base HEAD {BASELINE}')[-1].strip()
print('Checked out:', HEAD)
print('Merge base:', MB)
assert HEAD == PINNED_SCIENCE_COMMIT
assert MB == BASELINE
%pip install -q "$repo"
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
os.chdir(repo)

In [ ]:
import pytest
from openplaque.lad_distal_endpoint_continuation_v1 import synthetic_distal_continuation_self_test
print('Synthetic self-test:', synthetic_distal_continuation_self_test())
rc = pytest.main(['-q', 'tests/test_lad_distal_endpoint_continuation_v1.py'])
assert rc == 0, f'pytest failed with code {rc}'

In [ ]:
required = [
    DRIVE_ROOT / 'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
    DRIVE_ROOT / 'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
    DRIVE_ROOT / 'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
    DRIVE_ROOT / 'Left_Coronary_Backbone_Branch_Discovery_v1/summary.json',
    DRIVE_ROOT / 'Left_Coronary_Backbone_Branch_Discovery_v1/additional_branch_01_s00_d3_f0.csv',
]
for p in required:
    assert p.exists(), f'Missing prerequisite: {p}'
print('Preflight prerequisites present.')

In [ ]:
from openplaque.lad_distal_endpoint_continuation_v1 import run
result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
print(json.dumps(result['summary'], indent=2, default=str))
print('Report:', result['report'])
print('ZIP:', result['zip'])